In [ ]:
### developing schemas

In [18]:
from typing import List
from pydantic import BaseModel, Field

import pydantic
import enum
import os
import logging
from copy import deepcopy
import glob
import pathlib
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely as shpy

In [34]:
import pydantic

In [9]:
df_loc = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\data\processed\processed_data.csv'
df = pd.read_csv( df_loc )
df.describe()


,emissions_quantity,emissions_factor,capacity,capacity_factor,activity,area,pop2020
count,8.849000e+03,8849.000000,8849.000000,8849.000000,8.849000e+03,8849.000000,8.849000e+03
mean,8.188977e+05,0.555308,432.930049,0.326641,1.349288e+06,21.620055,1.284123e+07
std,1.618513e+06,0.248094,605.136630,0.095497,2.050491e+06,16.722962,1.145784e+07
min,2.000000e+03,0.308000,2.000000,0.114000,2.000000e+03,0.018377,5.768510e+05
25%,5.200000e+04,0.441000,39.000000,0.324000,1.000000e+05,11.617489,4.657757e+06
50%,2.230000e+05,0.452000,149.000000,0.331000,4.300000e+05,15.407903,8.631393e+06
75%,7.700000e+05,0.682000,628.000000,0.345000,1.862000e+06,25.539510,2.020125e+07
max,1.467300e+07,1.487000,4330.000000,0.694000,1.572700e+07,65.363350,3.953822e+07


In [30]:
df.columns

Index(['start_time', 'end_time', 'emissions_quantity', 'emissions_factor',
       'capacity', 'capacity_factor', 'activity', 'modified_date',
       'source_type', 'state', 'area', 'pop2020'],
      dtype='object')

In [ ]:
def load_us_states():
    states_file = pathlib.Path('us_states.txt')
    state_ls = states_file.read_text().splitlines()
    state_dictn = {   e_state.replace(' ', '_'): e_state  for e_state in state_ls   }
    return state_dictn

USState = enum.Enum( 'USState', load_us_states(), type= str )


In [32]:
filepath = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\api\us_states.txt'

states_file = pathlib.Path( filepath )
states_ls = states_file.read_text().splitlines()

state_dictn = { e_state.replace( ' ', '_'): e_state  for e_state in states_ls }

USState = enum.Enum( 'USState', state_dictn, type= str )





In [ ]:
class SourceType( str, enum.Enum ):
    gas = 'gas'
    oil = 'oil'
    coal = 'coal'
    other_fossil = 'other_fossil'
    biomass = 'biomass'
    waste = 'waste'

class Emission_Prediction_request( pydantic.BaseModel ):
     capacity: float = Field( ..., ge= 0, description= 'Capacity of the Industry - must be non negative numerical value' )
     capacity_factor: float = Field( ..., ge= 0, description= 'Capacity factor of the Industry - must be non negative numerical value' )
     activity: float = Field( ..., ge= 0, description= 'Activity of the Industry - must be non negative numerical value' )
     source_type: SourceType
     state: USState
     area: float = Field( ..., ge= 0, description= 'Area of the State - must be non negative numerical value' )
     pop: int = Field( ..., ge= 0, description= 'Population of the State - must be non negative integer value' )


class Emission_Prediction_response( pydantic.BaseModel ):
     predicted_emission: float 
     confidence_interval: List[ float ]
     feature_importance: dict
     prediction_time: str


<SourceType.other_fossil: 'other_fossil'>

In [11]:
df['source_type'].unique()

array(['gas', 'oil', 'coal', 'other_fossil', 'biomass', 'waste'],
      dtype=object)

In [ ]:
### inference

In [35]:
import joblib

import pathlib
import pandas as pd
import numpy as np

In [37]:
MODEL_PATH = 'models/trained/greenhouse_emission_predict_model.pkl' 
MODEL_PATH = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\models\trained\greenhouse_emission_predict_model.pkl'

PREPROCESSOR_PATH = 'models/trained/preprocessor.pkl'
PREPROCESSOR_PATH = r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\models\trained\preprocessor.pkl'

In [ ]:
model = joblib.load( MODEL_PATH )
preprocessor = joblib.load( PREPROCESSOR_PATH )



In [38]:

PREPROCESSOR_PATH

'E:\\Learning_course\\MLOps\\coursera_packt\\greenHouse_Emission_predictor\\models\\trained\\preprocessor.pkl'